In [ ]:
!git clone https://github.com/winddori2002/MANNER.git
%cd MANNER
!pip install -q pesq pystoi pyyaml tqdm

In [ ]:
s = open("src/metric.py").read()
s = s.replace("import librosa\n", "")
open("src/metric.py", "w").write(s)
print("patched metric.py (removed unused librosa import)")

In [ ]:
import re

for f in ["src/utils.py", "src/evaluation.py"]:
    s = open(f).read()
    s = s.replace("import neptune", "try:\n    import neptune\nexcept ImportError:\n    neptune = None")
    open(f, "w").write(s)

s = open("src/utils.py").read()
s = s.replace(
    '''def neptune_load(PARAMS):
    """
    logging: write your neptune account/project, api topken
    """
    neptune.init(\'ID/Project\', api_token = \'api-key\')
    neptune.create_experiment(name=PARAMS[\'ex_name\'], params=PARAMS)''',
    '''def neptune_load(PARAMS):
    """
    logging: write your neptune account/project, api topken
    """
    if neptune is None:
        print("neptune khong duoc cai / khong duoc cau hinh - bo qua logging")
        return
    neptune.init(\'ID/Project\', api_token = \'api-key\')
    neptune.create_experiment(name=PARAMS[\'ex_name\'], params=PARAMS)'''
)
open("src/utils.py", "w").write(s)
print("patched neptune import + neptune_load safe-guard")

In [ ]:
old = '''def train_select(valid_list, json_list):
    
    new_json = []
    for file in json_list:
        if (\'p\'+str(valid_list[0])+\'_\' not in file[0]) and (\'p\'+str(valid_list[1])+\'_\' not in file[0]) :
            new_json.append(file)
            
    return new_json
        
def valid_select(valid_list, json_list):
    
    new_json = []
    for file in json_list:
        if (\'p\'+str(valid_list[0])+\'_\' in file[0]) or (\'p\'+str(valid_list[1])+\'_\' in file[0]) :
            new_json.append(file)
                
    return new_json'''

new = '''import hashlib

def _val_bucket(path, mod=5):
    name = os.path.basename(str(path))
    h = hashlib.md5(name.encode()).hexdigest()
    return int(h, 16) % mod

def train_select(valid_list, json_list):
    return [f for f in json_list if _val_bucket(f[0]) != 0]

def valid_select(valid_list, json_list):
    return [f for f in json_list if _val_bucket(f[0]) == 0]'''

s = open("src/dataset.py").read()
assert old in s, "khong tim thay block can patch - kiem tra lai file"

old_load = '''            num_frames = 0
            offset     = 0
            if self.length is not None:
                offset     = self.stride * index
                num_frames = self.length
            out, sr    = torchaudio.load(str(file), offset=offset, num_frames=num_frames)'''
new_load = '''            if self.length is not None:
                offset     = self.stride * index
                num_frames = self.length
                out, sr    = torchaudio.load(str(file), frame_offset=offset, num_frames=num_frames)
            else:
                num_frames = 0
                out, sr    = torchaudio.load(str(file))'''
assert old_load in s, "khong tim thay block torchaudio.load can patch"
s = s.replace(old_load, new_load)

open("src/dataset.py", "w").write(s)
print("patched train/val split + torchaudio.load (frame_offset + length=None case)")

In [ ]:
patch = '''
def _safe_load_state(model, ckpt_path, device="cpu"):
    ckpt = torch.load(ckpt_path, map_location=device)
    state = ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt
    state = { (k[7:] if k.startswith("module.") else k): v for k, v in state.items() }
    model.load_state_dict(state)
    return ckpt
'''

for fname in ["src/train.py", "src/evaluation.py"]:
    s = open(fname).read()
    if "_safe_load_state" not in s:
        s = s.replace("import torch", "import torch\n" + patch, 1)
        open(fname, "w").write(s)

s = open("src/train.py").read()
s = s.replace(
    '''    def _load_checkpoint(self):
        checkpoint = torch.load(self.args.model_path + self.args.model_name)
        self.model.load_state_dict(checkpoint[\'state_dict\'])
        self.optimizer.load_state_dict(checkpoint[\'optimizer\']) 
        print(\'---load previous weigths and optimizer---\')''',
    '''    def _load_checkpoint(self):
        ckpt = _safe_load_state(self.model, self.args.model_path + self.args.model_name, self.args.device)
        if isinstance(ckpt, dict) and \'optimizer\' in ckpt:
            try:
                self.optimizer.load_state_dict(ckpt[\'optimizer\'])
            except Exception as e:
                print(\'optimizer state not restored:\', e)
        print(\'---load previous weights (optimizer if available)---\')'''
)
open("src/train.py", "w").write(s)

# evaluation.py: dung ham an toan khi test
s = open("src/evaluation.py").read()
s = s.replace(
    '''        checkpoint = torch.load(self.args.model_path + self.args.model_name)
        self.model.load_state_dict(checkpoint[\'state_dict\'])''',
    '''        _safe_load_state(self.model, self.args.model_path + self.args.model_name, self.args.device)'''
)
open("src/evaluation.py", "w").write(s)
print("patched checkpoint loading")

In [ ]:
s = open("src/stft_loss.py").read()
s = s.replace(
    '''    x_stft = torch.stft(x, fft_size, hop_size, win_length, window)
    real   = x_stft[..., 0]
    imag   = x_stft[..., 1]''',
    '''    x_stft = torch.stft(x, fft_size, hop_size, win_length, window, return_complex=True)
    real   = x_stft.real
    imag   = x_stft.imag'''
)
open("src/stft_loss.py", "w").write(s)
print("patched torch.stft return_complex")

In [ ]:
s = open("src/train.py").read()
old = """        if valid:
            return total_loss/(i+1), total_pesq/total_cnt, total_stoi/total_cnt
        else:
            return total_loss/(i+1)"""
new = """        if valid:
            if total_cnt == 0:
                print('CANH BAO: val_loader rong, tra ve 0 cho pesq/stoi')
                return 0, 0, 0
            return total_loss/(i+1), total_pesq/total_cnt, total_stoi/total_cnt
        else:
            return total_loss/(i+1)"""
assert old in s, "khong tim thay block can patch"
s = s.replace(old, new)
open("src/train.py", "w").write(s)
print("patched _run_epoch guard for empty val")

In [ ]:
!mkdir -p weights
!wget -q -O weights/manner_base_en.pth "https://huggingface.co/KhaBui/PESEM-VS/resolve/main/MANNER_EN.pth"
!cp weights/manner_base_en.pth weights/manner_base_en_ORIGINAL_BACKUP.pth
!ls -la weights

In [ ]:
import os, json, hashlib, torchaudio
from os.path import join as opj
from tqdm import tqdm

CLEAN_DIR = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/CLEAN"
NOISE_DIR = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/NOISE"

clean_list = sorted(os.listdir(CLEAN_DIR))
noise_list = sorted(os.listdir(NOISE_DIR))
assert len(clean_list) == len(noise_list), f"{len(clean_list)} clean vs {len(noise_list)} noise - so luong file khong khop"

def bucket10(name):
    h = hashlib.md5(name.encode()).hexdigest()
    return int(h, 16) % 10

def build(clean_files, noise_files, clean_dir, noise_dir, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    clean_info, noise_info = [], []
    for cf, nf in tqdm(list(zip(clean_files, noise_files))):
        cf_path = opj(clean_dir, cf); nf_path = opj(noise_dir, nf)
        clean_info.append([cf_path, torchaudio.load(cf_path)[0].shape[-1]])
        noise_info.append([nf_path, torchaudio.load(nf_path)[0].shape[-1]])
    json.dump(clean_info, open(opj(out_dir, "clean.json"), "w"), indent=2)
    json.dump(noise_info, open(opj(out_dir, "noisy.json"), "w"), indent=2)

train_c, train_n, test_c, test_n = [], [], [], []
for cf, nf in zip(clean_list, noise_list):
    if bucket10(cf) == 0:          # ~10% giu lai lam test that (chua tung thay khi train)
        test_c.append(cf); test_n.append(nf)
    else:
        train_c.append(cf); train_n.append(nf)

print(f"train pool: {len(train_c)} | held-out test: {len(test_c)}")

build(train_c, train_n, CLEAN_DIR, NOISE_DIR, "data_path/train")
build(test_c, test_n, CLEAN_DIR, NOISE_DIR, "data_path/test")

In [ ]:
!nvidia-smi

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!ls /kaggle/working/MANNER
!ls /kaggle/working/MANNER/weights
!ls /kaggle/working/MANNER/data_path/train | head -5

In [ ]:
%cd /kaggle/working/MANNER
!python main.py train \
  --train data_path/train --test data_path/test \
  --checkpoint True --model_path weights/ --model_name manner_base_en.pth \
  --learning_rate 1e-5 --epoch 20 --batch_size 6 --num_worker 4 \
  --set_stride 4 --device cuda:0

## Inference (chay Tester.test() voi save_enhanced=True)

`main.py` cua MANNER da co san che do `test` va co the luu enhanced wav neu bat `--save_enhanced True`. Checkpoint `weights/manner_base_en.pth` da bi ghi de bang trong so tot nhat sau khi finetune (xem `src/train.py`, `torch.save(checkpoint, model_path+model_name)`), nen chi can chay lai `main.py test` la du de lay ket qua enhanced tren tap test.

> Luu y: `--test` dang tro toi `data_path/test`, la 10% giu ngau nhien tu pool TRAIN (xem cell build data o tren), khong phai tap TEST chinh thuc (S01-S02) dung trong bai bao. Neu ban co san json/thu muc cho tap TEST chinh thuc, hay tro `--test` sang duong dan do de ket qua khop voi Table II/III trong paper.

In [ ]:
%cd /kaggle/working/MANNER
!python main.py test \
  --test data_path/test \
  --model_path weights/ --model_name manner_base_en.pth \
  --save_enhanced True --enhanced_path enhanced_test \
  --device cuda:0


In [ ]:
# write_result() luu 2 file cho moi utterance: <ten>_noise.wav (input) va <ten>_enhanced.wav (ket qua)
# -> tach rieng va doi ten enhanced ve dung ten file goc de khop voi CLEAN test khi tinh metrics.
import os, shutil

RAW_ENHANCED_DIR = "/kaggle/working/MANNER/enhanced_test"
FINAL_ENHANCED_DIR = "/kaggle/working/enhanced_MANNER"
os.makedirs(FINAL_ENHANCED_DIR, exist_ok=True)

assert os.path.isdir(RAW_ENHANCED_DIR), (
    f"Khong thay {RAW_ENHANCED_DIR}. Kiem tra lai cell test o tren da chay thanh cong voi save_enhanced=True chua."
)

count = 0
for fname in sorted(os.listdir(RAW_ENHANCED_DIR)):
    if not fname.endswith("_enhanced.wav"):
        continue
    orig_name = fname[: -len("_enhanced.wav")] + ".wav"
    shutil.copy(
        os.path.join(RAW_ENHANCED_DIR, fname),
        os.path.join(FINAL_ENHANCED_DIR, orig_name),
    )
    count += 1

print(f"Da doi ten va copy {count} file enhanced sang: {FINAL_ENHANCED_DIR}")
print("Buoc tiep theo: chay volume.py de RMS-normalize ve -16 dBFS, roi metrics.py de tinh PESQ/STOI/F0-RMSE/PFR.")
